In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

proxy = 'http://10.20.38.38:7890'
os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
import sys
sys.path.append('/home/ldy/Closed_loop_optimizing/model')
import torch
import numpy as np
from PIL import Image
from torchvision import transforms
import torchvision.models as models
import torch.nn as nn
from einops.layers.torch import Rearrange
import math
import open_clip
import torch.nn.functional as F
from torch.utils.data import DataLoader
from diffusion_prior import *
# from custom_pipeline import *
import random
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
# from util import (load_model_endocer, preprocess_image, generate_eeg, visualize_images, visualize_top_images, plot_similarity_and_mse_with_dual_axis, get_gteeg, save_eeg, load_thingstestimagedata, get_image_path, save_amx_similarities, save_results, 
# plot_similarity_range, save_value_function_to_txt)
from mne.time_frequency import psd_array_multitaper
from util import ( get_gteeg, save_eeg, get_image_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
image_folder = '/mnt/dataset0/ldy/4090_Workspace/4090_THINGS/images_set/test_images'

text_list = sorted(os.listdir(image_folder))
text_list = [file for file in text_list if not file.startswith('.')]
text_list

['00001_aircraft_carrier',
 '00002_antelope',
 '00003_backscratcher',
 '00004_balance_beam',
 '00005_banana',
 '00006_baseball_bat',
 '00007_basil',
 '00008_basketball',
 '00009_bassoon',
 '00010_baton4',
 '00011_batter',
 '00012_beaver',
 '00013_bench',
 '00014_bike',
 '00015_birthday_cake',
 '00016_blowtorch',
 '00017_boat',
 '00018_bok_choy',
 '00019_bonnet',
 '00020_bottle_opener',
 '00021_brace',
 '00022_bread',
 '00023_breadbox',
 '00024_bug',
 '00025_buggy',
 '00026_bullet',
 '00027_bun',
 '00028_bush',
 '00029_calamari',
 '00030_candlestick',
 '00031_cart',
 '00032_cashew',
 '00033_cat',
 '00034_caterpillar',
 '00035_cd_player',
 '00036_chain',
 '00037_chaps',
 '00038_cheese',
 '00039_cheetah',
 '00040_chest2',
 '00041_chime',
 '00042_chopsticks',
 '00043_cleat',
 '00044_cleaver',
 '00045_coat',
 '00046_cobra',
 '00047_coconut',
 '00048_coffee_bean',
 '00049_coffeemaker',
 '00050_cookie',
 '00051_cordon_bleu',
 '00052_coverall',
 '00053_crab',
 '00054_creme_brulee',
 '00055_cre

In [3]:

image_gt_folder = ['/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00014_bike/bike_14s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00181_television/television_14n.jpg'
                        , '/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00177_t-shirt/t-shirt_13s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00135_pie/pie_15s.jpg'
                        ,'/mnt/dataset0/ldy/datasets/THINGS_MEG/images_set/test_images/00131_pear/pear_13s.jpg']

image_gt_path = image_gt_folder[2]
dir_name = os.path.basename(os.path.dirname(image_gt_path))  # '00014_bike'
gt_category_id = text_list.index(dir_name) 
gt_category_id

176

In [4]:
dnn = 'alexnet' #'alexnet' #,'cornet_s'
sub = 'sub-01'
# encoder_model_path = f'/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/kyw/closed-loop/EEG-encoding/EEG_encoder/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-{dnn}/modeled_time_points-all/pretrained-True/lr-1e-05__wd-0e+00__bs-064/model_state_dict.pt'
encoder_model_path = f'/mnt/dataset0/kyw/closed-loop/EEG-encoding/EEG_encoder/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-{dnn}/modeled_time_points-all/pretrained-True/lr-1e-05__wd-0e+00__bs-064/model_state_dict.pt'
gt_eeg_folder = f'/mnt/dataset0/kyw/closed-loop/syn_eeg_gt'
# gt_eeg_folder = f'/mnt/dataset1/ldy/Workspace/Closed_loop_optimizing/data/syn_eeg_gt'

In [5]:
seed = 4  # 你可以根据需要更改随机种子
max_iterations = 50
gamma = 0.9
num_images= 10

os.makedirs(gt_eeg_folder, exist_ok=True)
gt_eeg_path = os.path.join(gt_eeg_folder, f"gt_{os.path.splitext(os.path.basename(image_gt_path))[0]}.npy")
print(f"gt_eeg_path {gt_eeg_path}")
# if os.path.exists(gt_eeg_path):
try:
    synthetic_eeg = torch.from_numpy(np.load(gt_eeg_path))
except:
    synthetic_eeg = torch.tensor(get_gteeg(image_gt_path, encoder_model_path, dnn, device)) 
gt_eeg_path = save_eeg(synthetic_eeg, gt_eeg_folder, file_name=f"gt_{os.path.splitext(os.path.basename(image_gt_path))[0]}.npy" )    

synthetic_eeg.shape

gt_eeg_path /mnt/dataset0/kyw/closed-loop/syn_eeg_gt/gt_t-shirt_13s.npy


torch.Size([17, 250])

In [6]:
import torch.nn.functional as F
def reward_function(psd, target_psd):    
    return F.cosine_similarity(target_psd, psd).item()

In [7]:
def load_target_psd(target_path, fs, selected_channel_idxes):
    target_signal = np.load(target_path, allow_pickle=True)
    selected_target_signal = target_signal[selected_channel_idxes, :]
    target_psd, _ = psd_array_multitaper(selected_target_signal, fs, adaptive=True, normalization='full', verbose=0)
    return torch.from_numpy(target_psd.flatten()).unsqueeze(0)

def load_target_psd_from_eeg(target_signal, fs, selected_channel_idxes):
    selected_target_signal = target_signal[selected_channel_idxes, :]
    target_psd, _ = psd_array_multitaper(selected_target_signal, fs, adaptive=True, normalization='full', verbose=0)
    return torch.from_numpy(target_psd.flatten()).unsqueeze(0)


In [8]:
clip_image_embeds = torch.load("/mnt/dataset1/ldy/Workspace/FLORA/data_preparing/ViT-H-14_features_test.pt")['img_features'].cpu()

/tmp/ipykernel_1441404/3186427432.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  clip_image_embeds = torch.load("/mnt/dataset1/ldy/Workspace/FLORA/data_preparing/ViT-H-

In [9]:

sub = 'sub-01'
model_path = f'/mnt/dataset0/jiahua/eeg_encoding/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-alexnet/modeled_time_points-all/pretrained-False/lr-1e-05__wd-0e+00__bs-064/model_state_dict.pt'
# target_eeg_path = f'/home/ldy/Closed_loop_optimizing/tjh/eeg_encoding/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-alexnet/modeled_time_points-all/pretrained-False/lr-1e-05__wd-0e+00__bs-064/gene_eeg/00183_tick_183.npy'
target_eeg_path = f'/home/ldy/Closed_loop_optimizing/tjh/eeg_encoding/results/{sub}/synthetic_eeg_data/encoding-end_to_end/dnn-alexnet/modeled_time_points-all/pretrained-False/lr-1e-05__wd-0e+00__bs-064/gene_eeg/00085_gondola_85.npy'

tar_psd_dir = f"/home/ldy/Closed_loop_optimizing/data/psd_feature/"
os.makedirs(tar_psd_dir, exist_ok = True)

file_path = os.path.join(tar_psd_dir, f"{dir_name}_psd_o1o2oz.pt")
if os.path.exists(file_path):
    target_psd = torch.load(file_path).cpu()            
else:
    target_psd = load_target_psd_from_eeg(synthetic_eeg, fs=250, selected_channel_idxes=[3, 4, 5])
    torch.save(target_psd, file_path)
    print(f"Files {file_path} saved!")    


Files /home/ldy/Closed_loop_optimizing/data/psd_feature/00177_t-shirt_psd_o1o2oz.pt saved!


In [10]:
image_folder = '/mnt/dataset0/ldy/4090_Workspace/4090_THINGS/images_set/test_images'
text_list = sorted(os.listdir(image_folder))
text_list = [file for file in text_list if not file.startswith('.')]

psd_dir = f"/home/ldy/Closed_loop_optimizing/data/psd_feature"
os.makedirs(psd_dir, exist_ok = True)
file_path = os.path.join(psd_dir, f"test_set_psd_features_o1o2oz.pt")

if os.path.exists(file_path):
    test_set_psd_features = torch.load(file_path).cpu()     
else:
    test_set_psd_features = []
    for folder in text_list:        
        image_files = os.listdir(os.path.join(image_folder, folder))
        image_path = os.path.join(os.path.join(image_folder, folder), image_files[0])
        eeg_signal = get_gteeg(image_path, encoder_model_path, dnn, device)
        psd = load_target_psd_from_eeg(eeg_signal.detach().cpu().numpy(), fs=250, selected_channel_idxes=[3, 4, 5])
        test_set_psd_features.append(psd)
    test_set_psd_features = torch.stack(test_set_psd_features)
    torch.save(test_set_psd_features, file_path)    

text_list

/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ldy/miniconda3/envs/BCI/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/home/ldy/Closed_loop_optimizing/experiments/util.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pyto

['00001_aircraft_carrier',
 '00002_antelope',
 '00003_backscratcher',
 '00004_balance_beam',
 '00005_banana',
 '00006_baseball_bat',
 '00007_basil',
 '00008_basketball',
 '00009_bassoon',
 '00010_baton4',
 '00011_batter',
 '00012_beaver',
 '00013_bench',
 '00014_bike',
 '00015_birthday_cake',
 '00016_blowtorch',
 '00017_boat',
 '00018_bok_choy',
 '00019_bonnet',
 '00020_bottle_opener',
 '00021_brace',
 '00022_bread',
 '00023_breadbox',
 '00024_bug',
 '00025_buggy',
 '00026_bullet',
 '00027_bun',
 '00028_bush',
 '00029_calamari',
 '00030_candlestick',
 '00031_cart',
 '00032_cashew',
 '00033_cat',
 '00034_caterpillar',
 '00035_cd_player',
 '00036_chain',
 '00037_chaps',
 '00038_cheese',
 '00039_cheetah',
 '00040_chest2',
 '00041_chime',
 '00042_chopsticks',
 '00043_cleat',
 '00044_cleaver',
 '00045_coat',
 '00046_cobra',
 '00047_coconut',
 '00048_coffee_bean',
 '00049_coffeemaker',
 '00050_cookie',
 '00051_cordon_bleu',
 '00052_coverall',
 '00053_crab',
 '00054_creme_brulee',
 '00055_cre

In [11]:
test_set_psd_features.shape

torch.Size([200, 1, 378])

In [12]:
target_psd.shape

torch.Size([1, 378])

In [13]:
# Initialize lists to store the data
data_x_list = []
data_y_list = []

for i, img_embed in enumerate(clip_image_embeds):    
    similarity = reward_function(test_set_psd_features[i], target_psd)
    print(f"similarity {similarity}")
    
    # Append the current data to the lists
    data_x_list.append(img_embed)
    data_y_list.append(-similarity*100)

# Convert lists to tensors and create the data dictionary
data = {
    "data_x": torch.stack(data_x_list),  # Stack image embeddings to form a batch
    "data_y": torch.tensor(data_y_list)  # Convert similarities to tensor
}

similarity 0.9973611247398648
similarity 0.9982830771589569
similarity 0.9939255843797188
similarity 0.991026344892308
similarity 0.9992320852840298
similarity 0.9984790395140613
similarity 0.9936744091189571
similarity 0.9951775674033927
similarity 0.9990277515491555
similarity 0.9991387586942156
similarity 0.994849244447022
similarity 0.9812575667476953
similarity 0.9759822860445502
similarity 0.9907698030178944
similarity 0.9905332709319679
similarity 0.9987236703102051
similarity 0.9985893089572031
similarity 0.9956611418020569
similarity 0.983761618132988
similarity 0.9938738373862408
similarity 0.9832894005676055
similarity 0.9967395847090161
similarity 0.9979045825772144
similarity 0.9931653714766814
similarity 0.9879202268316805
similarity 0.9985295145724489
similarity 0.993186199786684
similarity 0.9950156852212831
similarity 0.9938747583538323
similarity 0.9983785870809962
similarity 0.9954180099153251
similarity 0.9759781574477353
similarity 0.9909698803702607
similarity 0.9

In [14]:
data_y_list

[-99.73611247398648,
 -99.82830771589569,
 -99.39255843797187,
 -99.1026344892308,
 -99.92320852840298,
 -99.84790395140612,
 -99.36744091189571,
 -99.51775674033927,
 -99.90277515491555,
 -99.91387586942156,
 -99.4849244447022,
 -98.12575667476952,
 -97.59822860445502,
 -99.07698030178945,
 -99.05332709319678,
 -99.87236703102052,
 -99.8589308957203,
 -99.56611418020569,
 -98.37616181329881,
 -99.38738373862408,
 -98.32894005676054,
 -99.67395847090161,
 -99.79045825772144,
 -99.31653714766814,
 -98.79202268316804,
 -99.85295145724488,
 -99.3186199786684,
 -99.50156852212831,
 -99.38747583538323,
 -99.83785870809963,
 -99.54180099153251,
 -97.59781574477353,
 -99.09698803702607,
 -99.8816085707373,
 -99.94572767214594,
 -99.89125599642117,
 -99.79837154134748,
 -98.3470316302401,
 -98.06792847616536,
 -99.70776083973409,
 -99.8915238965214,
 -99.08053768716537,
 -99.75026947382601,
 -99.94620231436511,
 -99.98557945805602,
 -99.41585746630281,
 -99.87292272119379,
 -99.3262244821426,


In [15]:
data_y_list.index(min(data_y_list))

180